In [103]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import os

# Reproducibility
np.random.seed(42)

In [104]:
# Pipe configurations
pipe_length   = 50.0
pipe_diameter = 0.2
pipe_radius   = 0.1

nodes_per_timestep = 146
timesteps          = 700

# Output directory
base_dir = Path(os.getcwd()).parent / "data" / "synthetic"
output_dir = base_dir / "normal_variations"
output_dir.mkdir(parents=True, exist_ok=True)
baseline = {
    'pressure':             19657.0,
    'pressure-coefficient': 32600.0,
    'density':              2700.0,
    'velocity-magnitude':   2.479,
    'x-velocity':           2.478,
    'y-velocity':           0.0017,
    'z-velocity':          -0.0013,
    'turb-kinetic-energy':  0.0363,
    'turb-diss-rate':       24.4,
    'wall-shear':           10.4,
    'tailings-vof':         0.400,
}

noise = {
    'pressure':             500.0,
    'pressure-coefficient': 800.0,
    'density':              0.0,
    'velocity-magnitude':   0.05,
    'x-velocity':           0.05,
    'y-velocity':           0.005,
    'z-velocity':           0.005,
    'turb-kinetic-energy':  0.002,
    'turb-diss-rate':       1.5,
    'wall-shear':           0.5,
    'tailings-vof':         0.005,
}

# Simulation variation parameters
normal_scenarios = [
    {"name": "v100_c095", "velocity_factor": 1.00, "concentration_factor": 0.95},
    {"name": "v100_c105", "velocity_factor": 1.00, "concentration_factor": 1.05},
    {"name": "v095_c100", "velocity_factor": 0.95, "concentration_factor": 1.00},
    {"name": "v105_c100", "velocity_factor": 1.05, "concentration_factor": 1.00},
    {"name": "v095_c095", "velocity_factor": 0.95, "concentration_factor": 0.95},
    {"name": "v095_c105", "velocity_factor": 0.95, "concentration_factor": 1.05},
    {"name": "v105_c095", "velocity_factor": 1.05, "concentration_factor": 0.95},
    {"name": "v105_c105", "velocity_factor": 1.05, "concentration_factor": 1.05},
    {"name": "v098_c098", "velocity_factor": 0.98, "concentration_factor": 0.98},
    {"name": "v102_c102", "velocity_factor": 1.02, "concentration_factor": 1.02},
    {"name": "v097_c103", "velocity_factor": 0.97, "concentration_factor": 1.03},
    {"name": "v103_c097", "velocity_factor": 1.03, "concentration_factor": 0.97},
    {"name": "v096_c104", "velocity_factor": 0.96, "concentration_factor": 1.04},
    {"name": "v104_c096", "velocity_factor": 1.04, "concentration_factor": 0.96},
]


# ------ 9. Configuration Summary ------
print("========== CONFIG SUMMARY ==========")
print(f"Pipe Length: {pipe_length}")
print(f"Pipe Diameter: {pipe_diameter}")
print(f"Nodes per Timestep: {nodes_per_timestep}")
print(f"Timesteps: {timesteps}")
print(f"Total Scenarios: {len(scenarios)}")
print(f"Output Directory: {base_dir}")
print("====================================")


========== CONFIG SUMMARY ==========
Pipe Length: 50.0
Pipe Diameter: 0.2
Nodes per Timestep: 146
Timesteps: 700
Total Scenarios: 14
Output Directory: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic


In [105]:
# Node co-ordinate generator
def generate_node_coordinates(n_nodes):
    np.random.seed(42)

    node_numbers = np.arange(1, n_nodes + 1)
    x_coords = np.linspace(0, pipe_length, n_nodes)

    angles = np.linspace(0, 2 * np.pi, n_nodes)
    radii = np.random.uniform(0, pipe_radius, n_nodes)

    y_coords = radii * np.cos(angles)
    z_coords = radii * np.sin(angles)

    return node_numbers, x_coords, y_coords, z_coords


def generate_normal_pressure_profile(x_coords):
    P_inlet = 40000.0
    P_outlet = 0.0

    pressure = P_inlet - (x_coords / pipe_length) * (P_inlet - P_outlet)
    return pressure


In [106]:
# Single timestep generator
def generate_normal_variation_timestep( timestep,node_numbers, x_coords, y_coords, z_coords, velocity_factor,concentration_factor):
    n = len(node_numbers)
    pressure = generate_normal_pressure_profile(x_coords)
    pressure += np.random.normal(0, noise["pressure"], n)
    density = np.full(n, 2700.0)
    vel_mag = baseline["velocity-magnitude"] * velocity_factor + np.random.normal(0, noise["velocity-magnitude"], n)
    x_vel   = baseline["x-velocity"] * velocity_factor + np.random.normal(0, noise["x-velocity"], n)
    y_vel   = baseline["y-velocity"] + np.random.normal(0, noise["y-velocity"], n)
    z_vel   = baseline["z-velocity"] + np.random.normal(0, noise["z-velocity"], n)
    tke = baseline["turb-kinetic-energy"] * (velocity_factor ** 2) + np.random.normal(0, noise["turb-kinetic-energy"], n)
    tdr = baseline["turb-diss-rate"] * (velocity_factor ** 3) + np.random.normal(0, noise["turb-diss-rate"], n)
    wall_shear = baseline["wall-shear"] * (velocity_factor ** 2) + np.random.normal(0, noise["wall-shear"], n)

    tailings = baseline["tailings-vof"] * concentration_factor + np.random.normal(0, noise["tailings-vof"], n)

    p_coeff = pressure * (66166 / 40473)

    # Physical clipping
    pressure    = np.clip(pressure, 0, 45000)
    vel_mag     = np.clip(vel_mag, 0, 3.5)
    x_vel       = np.clip(x_vel, 0, 3.5)
    tke         = np.clip(tke, 0.023, 0.05)
    tdr         = np.clip(tdr, 0.07, 90.0)
    wall_shear  = np.clip(wall_shear, 0, 50.0)
    tailings    = np.clip(tailings, 0.163, 0.508)

    df = pd.DataFrame({
        "nodenumber": node_numbers,
        "x-coordinate": x_coords,
        "y-coordinate": y_coords,
        "z-coordinate": z_coords,
        "pressure": pressure,
        "pressure-coefficient": p_coeff,
        "density": density,
        "velocity-magnitude": vel_mag,
        "x-velocity": x_vel,
        "y-velocity": y_vel,
        "z-velocity": z_vel,
        "turb-kinetic-energy": tke,
        "turb-diss-rate": tdr,
        "wall-shear": wall_shear,
        "tailings-vof": tailings,
        "timestep": np.full(n, timestep),
        "label": np.zeros(n, dtype=int)
    })

    return df

In [107]:
# Full scenario generator
def generate_normal_variation_scenario( scenario_name, velocity_factor,concentration_factor,save=True):

    node_numbers, x_coords, y_coords, z_coords = generate_node_coordinates(nodes_per_timestep)

    all_timesteps = []
    for t in range(1, timesteps + 1):

        df_t = generate_normal_variation_timestep(
            t,
            node_numbers,
            x_coords,
            y_coords,
            z_coords,
            velocity_factor,
            concentration_factor
        )

        all_timesteps.append(df_t)

        if t % 100 == 0:
            print(f"{scenario_name} → timestep {t}/{timesteps}")
    df_scenario = pd.concat(all_timesteps, ignore_index=True)

    expected_shape = (nodes_per_timestep * timesteps, 17)

    assert df_scenario.shape == expected_shape, \
        f"Shape mismatch: got {df_scenario.shape}, expected {expected_shape}"

    assert (df_scenario["label"] == 0).all(), "Label check failed"
    if save:
        filename = f"normal_{scenario_name}.csv"
        filepath = output_dir / filename
        df_scenario.to_csv(filepath, index=False)
        print(f"Saved → {filepath}")

    return df_scenario



In [108]:
normal_scenarios = [
    {"name": "v100_c095", "vel": 1.00, "conc": 0.95},
    {"name": "v100_c105", "vel": 1.00, "conc": 1.05},
    {"name": "v095_c100", "vel": 0.95, "conc": 1.00},
    {"name": "v105_c100", "vel": 1.05, "conc": 1.00},
    {"name": "v095_c095", "vel": 0.95, "conc": 0.95},
    {"name": "v095_c105", "vel": 0.95, "conc": 1.05},
    {"name": "v105_c095", "vel": 1.05, "conc": 0.95},
    {"name": "v105_c105", "vel": 1.05, "conc": 1.05},
    {"name": "v098_c098", "vel": 0.98, "conc": 0.98},
    {"name": "v102_c102", "vel": 1.02, "conc": 1.02},
    {"name": "v097_c103", "vel": 0.97, "conc": 1.03},
    {"name": "v103_c097", "vel": 1.03, "conc": 0.97},
    {"name": "v096_c104", "vel": 0.96, "conc": 1.04},
    {"name": "v104_c096", "vel": 1.04, "conc": 0.96},
]
normal_dataframes = {}

for scenario in normal_scenarios:

    name = scenario["name"]
    vel = scenario["vel"]
    conc = scenario["conc"]

    print(f"\nScenario: {name}")
    print(f"  velocity_factor = {vel}")
    print(f"  concentration_factor = {conc}")

    df = generate_normal_variation_scenario(
        scenario_name=name,
        velocity_factor=vel,
        concentration_factor=conc,
        save=True
    )

    normal_dataframes[name] = df
print(f"Total scenarios generated: {len(normal_dataframes)}")


Scenario: v100_c095
  velocity_factor = 1.0
  concentration_factor = 0.95
v100_c095 → timestep 100/700
v100_c095 → timestep 200/700
v100_c095 → timestep 300/700
v100_c095 → timestep 400/700
v100_c095 → timestep 500/700
v100_c095 → timestep 600/700
v100_c095 → timestep 700/700
Saved → /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/normal_variations/normal_v100_c095.csv

Scenario: v100_c105
  velocity_factor = 1.0
  concentration_factor = 1.05
v100_c105 → timestep 100/700
v100_c105 → timestep 200/700
v100_c105 → timestep 300/700
v100_c105 → timestep 400/700
v100_c105 → timestep 500/700
v100_c105 → timestep 600/700
v100_c105 → timestep 700/700
Saved → /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/normal_variations/normal_v100_c105.csv

Scenario: v095_c100
  velocity_factor = 0.95
  concentration_factor = 1.0
v095_c100 → timestep 100/700
v095_c100 → timestep 200/700
v095_c100 → timestep 300/700
v095_c100 → timestep 4

In [109]:
print("Normal variation physics check")
scenario_1 = normal_dataframes["v095_c105"]
scenario_2 = normal_dataframes["v105_c095"]

def get_timestep(df, t):
    return df[df["timestep"] == t]

df_s1_t350 = get_timestep(scenario_1, 350)
df_s2_t350 = get_timestep(scenario_2, 350)

baseline_pressure = baseline["pressure"]
baseline_velocity = baseline["velocity-magnitude"]
baseline_tailings = baseline["tailings-vof"]

print("\n--- Scenario: v095_c105 ---")
print("Mean pressure:", df_s1_t350["pressure"].mean())
print("Mean velocity:", df_s1_t350["velocity-magnitude"].mean())
print("Mean tailings-vof:", df_s1_t350["tailings-vof"].mean())

print("\n--- Scenario: v105_c095 ---")
print("Mean pressure:", df_s2_t350["pressure"].mean())
print("Mean velocity:", df_s2_t350["velocity-magnitude"].mean())
print("Mean tailings-vof:", df_s2_t350["tailings-vof"].mean())

print("\nEXPECTED CHECKS:")
print("- Pressure ≈ baseline (unchanged)")
print("- v095_c105: velocity decreased by ~5%, tailings ↑ ~5%")
print("- v105_c095: velocity increased by ~5%, tailings ↓ ~5%")
print("- All values within physical bounds")
print("- No fault signatures present")

Normal variation physics check

--- Scenario: v095_c105 ---
Mean pressure: 19971.68357540961
Mean velocity: 2.3516234142108314
Mean tailings-vof: 0.4197941409658399

--- Scenario: v105_c095 ---
Mean pressure: 19971.68357540961
Mean velocity: 2.599523414210832
Mean tailings-vof: 0.3797941409658399

EXPECTED CHECKS:
- Pressure ≈ baseline (unchanged)
- v095_c105: velocity decreased by ~5%, tailings ↑ ~5%
- v105_c095: velocity increased by ~5%, tailings ↓ ~5%
- All values within physical bounds
- No fault signatures present


In [110]:

data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "raw")
normal_file_path = os.path.join(data_path, "normal_dataset.csv")

df_original_normal = pd.read_csv(normal_file_path)
# Display the first few rows of the dataset
df_original_normal.head(10)

,nodenumber,x-coordinate,y-coordinate,z-coordinate,pressure,pressure-coefficient,density,velocity-magnitude,x-velocity,y-velocity,z-velocity,turb-kinetic-energy,turb-diss-rate,wall-shear,tailings-vof,timestep,label
0,1,25.000001,0.008344,-0.097383,21168.812960,34653.963120,2700.00006,0.000000,0.000000,0.000000,0.000000,0.036400,0.577750,38.203626,1.844108e-18,1,0
1,2,0.000000,0.012623,-0.096541,38814.917600,64472.464840,2700.00000,0.000000,0.000000,0.000000,0.000000,0.045525,44.802546,40.826937,4.000014e-01,1,0
2,3,50.000000,0.012623,-0.096541,1251.180643,2510.182354,2700.00000,0.000000,0.000000,0.000000,0.000000,0.039119,0.584892,40.845045,2.189194e-143,1,0
3,4,25.000000,-0.023607,-0.094895,20966.104920,34438.657840,2700.00000,0.000000,0.000000,0.000000,0.000000,0.036468,0.621301,38.334753,1.844973e-18,1,0
4,5,0.000000,-0.033333,-0.090946,38685.698180,64330.871500,2700.00000,0.000000,0.000000,0.000000,0.000000,0.045570,43.239824,40.693794,4.000014e-01,1,0
5,6,50.000000,-0.033333,-0.090946,1354.487389,2514.356787,2700.00000,0.000000,0.000000,0.000000,0.000000,0.038138,0.572188,40.550091,1.797694e-143,1,0
6,7,25.000001,-0.022285,-0.090455,21058.807410,34381.726390,2700.00000,3.263889,3.263316,-0.031496,-0.031253,0.036513,0.310620,0.000000,1.904717e-18,1,0
7,8,25.000001,0.006244,-0.085583,21199.150160,34610.857400,2700.00000,3.283000,3.282915,-0.005660,-0.016153,0.036214,0.284167,0.000000,1.917765e-18,1,0
8,9,50.000000,0.015056,-0.082753,0.000000,2094.493977,2700.00000,3.347043,3.340350,-0.029204,0.194177,0.036965,0.326557,0.000000,2.162621e-143,1,0
9,10,0.000000,0.015415,-0.081318,39563.257460,64880.611580,2700.00000,3.500000,3.500000,0.000000,0.000000,0.045937,84.647029,0.000000,4.000000e-01,1,0


In [111]:
print("========== COMBINING NORMAL DATASETS ==========")

assert df_original_normal["label"].eq(0).all(), "Original dataset contains non-normal labels"
assert df_original_normal.shape[1] == 17, "Column mismatch in original dataset"


print("Original dataset shape:", df_original_normal.shape)

all_dfs = list(normal_dataframes.values())
all_dfs.append(df_original_normal)

df_all_normal = pd.concat(all_dfs, ignore_index=True)

expected_rows = 102200 * 15  # 14 new + 1 original
expected_shape = (expected_rows, 17)

assert df_all_normal.shape == expected_shape, f"Shape mismatch: {df_all_normal.shape}"

print("Final shape:", df_all_normal.shape)
print("Label distribution:\n", df_all_normal["label"].value_counts())
save_path = output_dir / "all_normal_combined.csv"
df_all_normal.to_csv(save_path, index=False)

print(f"\nSaved combined dataset → {save_path}")

========== COMBINING NORMAL DATASETS ==========
Original dataset shape: (102200, 17)
Final shape: (1533000, 17)
Label distribution:
 label
0    1533000
Name: count, dtype: int64

Saved combined dataset → /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/normal_variations/all_normal_combined.csv
